### This notebook computes a set of indicators from the WaterALLOC database for the 27 basins of IKI Project

**List of Indicators:**
- P13. Water yield: Water production per basin and per square kilometer
- VSS6. Volume of annual water demand for demographic use
- VSS9. Supply reliability: Portion of time during which the water demand is fully met for the agricultural sector
- VSS10. Supply reliability: Portion of time during which the water demand is fully met for the municipal sector
- VSS11. Average deficit, Average magnitude of the supply deficit in the agricultural sector
- VSS12. Average deficit, Average magnitude of the supply deficit in the municipal sector
- VSS13_A, VSS13_P. Index of hydrological stress for the agriculture sector (A) and the municipal sector (P)
- VSB10. Availability of Water by Basin for the Agricultural Sector
- VCA16_A, VCA16_P. Relative frequency of recovery after failure for the agricultural sector (A) and the municiapl sector (P)
- VCA17_A, VCA17_P, VCA17_E. Percentage of net imported water volume relative to the total demand for the agricultural sector (A), the municipal sector (P), and the energy sector (E)
- VCA18. Index of volume of reservoir storage

**Created:** 1/22/2026 by Sophia Bakar (sbakar@rti.org) and Enrique Triana (etriana@rti.org)



In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import sqlite3
import matplotlib.pyplot as plt
import re 
import yaml
from pathlib import Path

#### Seleccion de rutas como funcion del usuario

In [2]:
# set master path for input data from config file

config_path = Path("../../config.yaml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

master_path = Path(config["master_path"])

In [3]:
subbasins_shapefile = master_path / "Modelacion" / "Grupos_Modelacion" / "GIS_WaterALLOC_General" / "Peru_AHD_with_districts.shp"
db_path = master_path / "Indicadores" / "BD_RiesgoClimatico_IKI.db"
#wateralloc_db = master_path / "Modelacion" / "Grupos_Modelacion" / "Resultados" / "BalanceHidrico.sqlite"
wateralloc_db = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"
wateralloc_db_2 = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico_2.sqlite"

In [4]:
wateralloc_dbs = [
    wateralloc_db,
    wateralloc_db_2
]

In [5]:
subbasins_gdf = gpd.read_file(subbasins_shapefile).set_index('COMID').to_crs('WGS84')

In [6]:
conn = sqlite3.connect(db_path)

## NEED A NEW APPROACH/WAY TO FLAG FOR THIS

indicators_df = pd.read_sql_query(
    """
    SELECT IndID, TextID
    FROM Indicators
    WHERE SIG_Type = 'Wateralloc DB'
    """,
    conn
)

conn.close()

In [7]:
indicators_df

,IndID,TextID
0,113,P13
1,310,VSB10
2,5161,VCA16_A
3,5162,VCA16_P
4,5171,VCA17_A
5,5172,VCA17_P
6,5173,VCA17_E
7,518,VCA18
8,406,VSS6
9,409,VSS9


In [8]:
# Connect to Indicators DB and get available scenarios
conn = sqlite3.connect(db_path)

wa_scenarios_df = pd.read_sql_query(
    """
    SELECT WaScnID, WaScnName
    FROM WaScenarios
    ORDER BY WaScnID
    """,
    conn
)

conn.close()

# For now: only baseline and first future
scenario_ids = wa_scenarios_df.loc[
    wa_scenarios_df['WaScnID'].isin([1, 3, 7]), 'WaScnID'
].tolist()

# for all scenarios:
#scenario_ids = wa_scenarios_df['WaScnID'].tolist()

wa_scenarios_df

,WaScnID,WaScnName
0,1,CC_CMIP6_85_2050
1,3,Linea_Base_2020
2,4,Linea_Base_2020_Embalses
3,7,CC_CMIP6_45_2050


### Process all the WaterALLOC Scenarios
Los escenarios en WaterALLOC deben corresponder a un escenario en la base de datos de indicadores. Es esta seccion procesamos todos los escenarios en la base de datos vinculando con el escenario correspondiente en la base de datos de indicadores.

Este calculo necesita un indicador unico en la base de datos de indicadores para las combinaciones de  escenarios de indicadores y WaterALLOC.  
#### Method
Using SQL we attach the indicators database and execute an insert query in the indicators database using the processed data from the WaterALLOC scenarios database.  

Only the COMIDs processed to the WaterALLOC database are available.  The risk calculation should handle the missing COMID values.

#### Calculation Notes
The indicator for each COMID is queried as the .  

In [9]:
def process_wateralloc_db(
    wateralloc_db_path,
    db_path,
    subbasins_gdf,
    indicators_df,
    scenario_map
):

    print("\n===================================================")
    print("Processing WaterALLOC database:")
    print(wateralloc_db_path)
    print("===================================================")

    conn_wa = sqlite3.connect(wateralloc_db_path)
    cursor = conn_wa.cursor()

    cursor.execute(fr"ATTACH DATABASE '{db_path}' AS RiesgoDB;")

    # Use only COMIDs present in this WaterALLOC DB
    wateralloc_comids_df = pd.read_sql_query(
        """
        SELECT DISTINCT COMID
        FROM "WAMSS_Balance por COMID (+Indice de estres)"
        WHERE COMID IS NOT NULL
        """,
        conn_wa
    )

    wateralloc_comids_df["COMID"] = pd.to_numeric(
    wateralloc_comids_df["COMID"],
    errors="coerce")

    wateralloc_comids_df = wateralloc_comids_df.dropna(subset=["COMID"]).copy()

    wateralloc_comids_df["COMID"] = wateralloc_comids_df["COMID"].astype(int)

    wateralloc_comids_df.to_sql(
        "WaterALLOC_COMIDs",
        conn_wa,
        if_exists="replace",
        index=False
    )

    print(f"WaterALLOC COMIDs in this DB: {len(wateralloc_comids_df)}")

    subbasins_area_df = (
        subbasins_gdf.reset_index()[["COMID", "AREASQKM"]]
        .astype({"COMID": int})
    )

    subbasins_area_df.to_sql(
        "SubbasinArea",
        conn_wa,
        if_exists="replace",
        index=False
    )

    # ------------------------------------------------------------
    # Create/update views
    # ------------------------------------------------------------

    cursor.execute("""DROP VIEW IF EXISTS "ImportExport fraction por COMID";""")

    cursor.execute("""
    CREATE VIEW "ImportExport fraction por COMID" AS 
    SELECT 
        a.RunID,
        a.Cuenca, 
        a.COMID, 
        SUM(importExport) AS NetImport,
        SUM(importExport) / AVG(TotAfluencia) AS ImportFract 
    FROM (
        SELECT 
            RunID,
            Cuenca,
            COMID, 
            AVG(Volumen) AS ImportExport
        FROM "WAMSS_Importacion y Exportacion anual por COMID"
        WHERE [Tipo] = 'Importacion'
        GROUP BY RunID, Cuenca, COMID

        UNION 

        SELECT 
            RunID,
            Cuenca,
            COMID, 
            AVG(-Volumen) AS ImportExport
        FROM "WAMSS_Importacion y Exportacion anual por COMID"
        WHERE [Tipo] = 'Exportacion'
        GROUP BY RunID, Cuenca, COMID
    ) AS a
    LEFT JOIN (
        SELECT 
            RunID,
            Cuenca,
            AVG(Afluencia) AS [TotAfluencia]
        FROM "WAMSS_Oferta anual por tipo por cuenca"
        GROUP BY RunID, Cuenca
    ) AS b 
        ON b.RunID = a.RunID 
       AND b.Cuenca = a.Cuenca 
    GROUP BY a.RunID, a.Cuenca, a.COMID
    """)

    cursor.execute("""DROP VIEW IF EXISTS "OfertaNorm por COMID";""")

    cursor.execute("""
    CREATE VIEW "OfertaNorm por COMID" AS
    SELECT 
        annual.RunID,
        annual.COMID,
        AVG(annual.OfertaTot_anual) / area.AREASQKM AS OfertaNorm
    FROM (
        SELECT 
            RunID,
            COMID,
            Año,
            SUM(Afluencia) AS OfertaTot_anual
        FROM "WAMSS_Oferta anual por tipo por COMID"
        WHERE TipoAfluencia <> 'Recarga'
        GROUP BY RunID, COMID, Año
    ) AS annual
    JOIN SubbasinArea area
        ON area.COMID = annual.COMID
    GROUP BY annual.RunID, annual.COMID
    """)

    cursor.execute("""DROP VIEW IF EXISTS "ReservoirStorage por COMID";""")

    cursor.execute("""
    CREATE VIEW "ReservoirStorage por COMID" AS
    SELECT 
        b.WaScnID,
        a.RunID,
        a.COMID,
        AVG(a.Almacenamiento) AS Storage_Medio
    FROM "WAMSS_Volumen promedio en embalses por COMID" a
    JOIN WAMMS_RunsInfo b 
        ON a.RunID = b.RunID
    GROUP BY b.WaScnID, a.RunID, a.COMID
    """)

    # ------------------------------------------------------------
    # Process scenarios and indicators
    # ------------------------------------------------------------

    for target_waScn_ID, source_waScn_ID in scenario_map.items():

        print(f"\nProcessing target WaScnID={target_waScn_ID} using source WaScnID={source_waScn_ID}")

        run_ids_list = [row[0] for row in cursor.execute(
            f"SELECT RunID FROM WAMMS_RunsInfo WHERE WaScnID = {source_waScn_ID}"
            ).fetchall()]

        print(f"\tRunIDs for this scenario: {run_ids_list}")

        if len(run_ids_list) == 0:
            print(f"\tNo RunIDs found for WaScnID={source_waScn_ID}. Skipping.")
            continue

        run_ids_sql = ",".join(map(str, run_ids_list))

        for ind_row in indicators_df.itertuples(index=False):

            indID = ind_row.IndID
            textID = ind_row.TextID

            print(f"\tProcessing indicator {textID} (IndID={indID})")

            default_fill = 1 if textID in ["VSS9", "VSS10"] else 0

            if textID == "VSS9":
                source_table = "[WAMSS_Confiabilidad y Deficit por COMID]"
                value_calc = "b.[Confiabilidad]"
                and_where = "AND b.Sector = 'Agrario'"

            elif textID == "VSB10":
                source_table = "[WAMSS_Balance por COMID (+Indice de estres)]"
                value_calc = "(b.[Oferta Local Sup] + b.[Oferta Entrada] - b.[Dem Local Sup])"
                and_where = ""

            elif textID == "VSS10":
                source_table = "[WAMSS_Confiabilidad y Deficit por COMID]"
                value_calc = "b.[Confiabilidad]"
                and_where = "AND b.Sector = 'Poblacional'"

            elif textID == "VSS11":
                source_table = "[WAMSS_Confiabilidad y Deficit por COMID]"
                value_calc = "b.[DeficitProm]"
                and_where = "AND b.Sector = 'Agrario'"

            elif textID == "VSS12":
                source_table = "[WAMSS_Confiabilidad y Deficit por COMID]"
                value_calc = "b.[DeficitProm]"
                and_where = "AND b.Sector = 'Poblacional'"

            elif textID in ["VSS13_A", "VSS13_P"]:
                source_table = "[WAMSS_Balance por COMID (+Indice de estres)]"
                value_calc = "b.[IndiceEstres_Sup]"
                and_where = ""

            elif textID in ["VCA16_A", "VCA16_P"]:
                source_table = "[WAMSS_Resiliencia por COMID y Sector]"
                value_calc = "b.[frequency_1_to_0]"
                and_where = (
                    "AND b.Sector = 'Agrario'"
                    if textID.endswith("_A")
                    else "AND b.Sector = 'Poblacional'"
                )

            elif textID in ["VCA17_A", "VCA17_P", "VCA17_E"]:
                source_table = "[ImportExport fraction por COMID]"
                value_calc = "b.[ImportFract]"
                and_where = ""

            elif textID == "VSS6":
                source_table = "[WAMSS_Demanda anual promedio por tipo de demanda por COMID]"
                value_calc = "CASE WHEN b.Demanda = 0 THEN 0 ELSE b.Suministro / b.Demanda END"
                and_where = "AND b.Sector = 'Poblacional'"

            elif textID == "P13":
                source_table = "[OfertaNorm por COMID]"
                value_calc = "b.[OfertaNorm]"
                and_where = ""

            elif textID == "VCA18":
                source_table = "[ReservoirStorage por COMID]"
                value_calc = "b.[Storage_Medio]"
                and_where = ""

            else:
                print(f"\t\tSkipping undefined indicator {textID}")
                continue

            count_source_rows = cursor.execute(f"""
                SELECT COUNT(*)
                FROM {source_table} AS b
                WHERE b.RunID IN ({run_ids_sql})
                {and_where};
            """).fetchone()[0]

            print(f"\t\tRows in source table for this indicator & scenario: {count_source_rows}")

            insert_query = f"""
                INSERT OR REPLACE INTO [RiesgoDB].IndValues_WaALLOC 
                    (WaScnID, IndID, COMID, Value)
                SELECT
                    {target_waScn_ID} AS WaScnID,
                    {indID} AS IndID,
                    c.COMID,
                    COALESCE(v.Value, {default_fill}) AS Value
                FROM WaterALLOC_COMIDs AS c
                LEFT JOIN (
                    SELECT
                        b.COMID,
                        AVG({value_calc}) AS Value
                    FROM {source_table} AS b
                    WHERE b.RunID IN ({run_ids_sql})
                    {and_where}
                    GROUP BY b.COMID
                ) AS v
                ON v.COMID = c.COMID;
            """

            cursor.execute(insert_query)

            n_rows = cursor.execute(f"""
                SELECT COUNT(*)
                FROM [RiesgoDB].IndValues_WaALLOC
                WHERE WaScnID = {target_waScn_ID}
                  AND IndID  = {indID};
            """).fetchone()[0]

            print(f"\t\tTotal rows now in RiesgoDB for this scenario & indicator: {n_rows}")

    conn_wa.commit()

    try:
        cursor.execute("DETACH DATABASE RiesgoDB;")
    except sqlite3.OperationalError:
        pass

    conn_wa.close()

    print(f"\nFinished processing: {wateralloc_db_path}")

In [10]:
wateralloc_db_configs = [
    {
        "path": wateralloc_db,
        "scenario_map": {
            1: 1,
            3: 3,
            7: 7
        }
    },
    {
        "path": wateralloc_db_2,
        "scenario_map": {
            1: 6,
            3: 3,
            7: 1
        }
    }
]

In [11]:
target_scenario_ids = [1, 3, 7]

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

for waScn_ID in target_scenario_ids:
    for ind_row in indicators_df.itertuples(index=False):
        cursor.execute(
            """
            DELETE FROM IndValues_WaALLOC
            WHERE WaScnID = ?
              AND IndID = ?;
            """,
            (waScn_ID, ind_row.IndID)
        )

conn.commit()
conn.close()

In [12]:
for wa_config in wateralloc_db_configs:
    process_wateralloc_db(
        wateralloc_db_path=wa_config["path"],
        db_path=db_path,
        subbasins_gdf=subbasins_gdf,
        indicators_df=indicators_df,
        scenario_map=wa_config["scenario_map"]
    )


Processing WaterALLOC database:
C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite
WaterALLOC COMIDs in this DB: 2405

Processing target WaScnID=1 using source WaScnID=1
	RunIDs for this scenario: [1, 4, 6, 11, 13, 17, 21]
	Processing indicator P13 (IndID=113)
		Rows in source table for this indicator & scenario: 1825
		Total rows now in RiesgoDB for this scenario & indicator: 2405
	Processing indicator VSB10 (IndID=310)
		Rows in source table for this indicator & scenario: 834972
		Total rows now in RiesgoDB for this scenario & indicator: 2405
	Processing indicator VCA16_A (IndID=5161)
		Rows in source table for this indicator & scenario: 762
		Total rows now in RiesgoDB for this scenario & indicator: 2405
	Processing indicator VCA16_P (IndID=5162)
		Rows in source table for this indicator & scenario: 708
		Total rows now in RiesgoDB for this scenario & indicator: 2405
	Processin